# 02 - Limpieza y transformacion

Este notebook prepara una tabla limpia de peliculas a partir de MovieLens. El objetivo es dejar los datos listos para analisis exploratorio y para pasos posteriores del recomendador, sin aplicar todavia ninguna tecnica de recomendacion.

## 1. Importacion de librerias

In [24]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

base_dir = Path('..').resolve()
sys.path.append(str(base_dir))

from src.data_utils import load_movies, load_ratings, load_tags
from src.preprocessing import (
    build_movies_clean,
    calculate_rating_summary,
    clean_title,
    create_genre_one_hot,
    extract_year_from_title,
    split_genres,
)

## 2. Carga de datos

In [25]:
# Definimos las rutas de entrada y cargamos los CSV originales.
raw_dir = base_dir / 'data' / 'raw'
processed_dir = base_dir / 'data' / 'processed'

movies_path = raw_dir / 'movies.csv'
ratings_path = raw_dir / 'ratings.csv'
tags_path = raw_dir / 'tags.csv'

movies_df = load_movies(movies_path)
ratings_df = load_ratings(ratings_path)
tags_df = load_tags(tags_path)

In [26]:
# Trabajamos sobre una copia para no tocar los datos originales.
movies_clean = movies_df.copy()

# Comprobamos las columnas clave.
print('movieId nulos:', movies_clean['movieId'].isna().sum())
print('title nulos:', movies_clean['title'].isna().sum())
print('genres nulos:', movies_clean['genres'].isna().sum())
print('filas duplicadas por movieId:', movies_clean['movieId'].duplicated().sum())
print("peliculas con '(no genres listed)':", (movies_clean['genres'] == '(no genres listed)').sum())

# Aplicamos la limpieza simple de generos y titulos.
movies_clean['genres'] = movies_clean['genres'].fillna('')
movies_clean.loc[movies_clean['genres'] == '(no genres listed)', 'genres'] = ''
movies_clean['year'] = movies_clean['title'].apply(extract_year_from_title)
movies_clean['title_clean'] = movies_clean['title'].apply(clean_title)
print('titulos sin ano detectado:', movies_clean['year'].isna().sum())

movieId nulos: 0
title nulos: 0
genres nulos: 0
filas duplicadas por movieId: 0
peliculas con '(no genres listed)': 7080
titulos sin ano detectado: 771


In [27]:
print('movies_df nulls')
display(movies_df.isnull().sum())

print('ratings_df nulls')
display(ratings_df.isnull().sum())

print('tags_df nulls')
display(tags_df.isnull().sum())

print('movies_df duplicated rows:', movies_df.duplicated().sum())
print('ratings_df duplicated rows:', ratings_df.duplicated().sum())
print('tags_df duplicated rows:', tags_df.duplicated().sum())

movies_df nulls


movieId    0
title      0
genres     0
dtype: int64

ratings_df nulls


userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

tags_df nulls


userId        0
movieId       0
tag          17
timestamp     0
dtype: int64

movies_df duplicated rows: 0
ratings_df duplicated rows: 0
tags_df duplicated rows: 0


## 4. Comprobacion de la logica de limpieza

In [28]:
# Probamos las funciones reutilizables sobre ejemplos sencillos.
print('A?o extraido:', extract_year_from_title('Toy Story (1995)'))
print('Titulo limpio:', clean_title('Toy Story (1995)'))
print('Generos separados:', split_genres('Adventure|Animation|Children|Comedy|Fantasy'))

A?o extraido: 1995
Titulo limpio: Toy Story
Generos separados: ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy']


## 5. Limpieza basica de `movies.csv`

In [29]:
# Trabajamos sobre una copia para no tocar los datos originales.
movies_clean = movies_df.copy()

# Comprobamos las columnas clave.
print('movieId nulos:', movies_clean['movieId'].isna().sum())
print('title nulos:', movies_clean['title'].isna().sum())
print('genres nulos:', movies_clean['genres'].isna().sum())
print('filas duplicadas por movieId:', movies_clean['movieId'].duplicated().sum())
print("peliculas con '(no genres listed)':", (movies_clean['genres'] == '(no genres listed)').sum())

# Aplicamos la limpieza simple de generos y titulos.
movies_clean['genres'] = movies_clean['genres'].fillna('')
movies_clean.loc[movies_clean['genres'] == '(no genres listed)', 'genres'] = ''
movies_clean['year'] = movies_clean['title'].apply(extract_year_from_title)
movies_clean['title_clean'] = movies_clean['title'].apply(clean_title)
print('titulos sin ano detectado:', movies_clean['year'].isna().sum())

movieId nulos: 0
title nulos: 0
genres nulos: 0
filas duplicadas por movieId: 0
peliculas con '(no genres listed)': 7080
titulos sin ano detectado: 771


## 6. Generos con one-hot encoding

In [30]:
# Creamos una tabla con una columna por genero.
movie_genres_encoded = create_genre_one_hot(movies_df)
display(movie_genres_encoded.head())
print('movie_genres_encoded shape:', movie_genres_encoded.shape)

,movieId,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,3,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


movie_genres_encoded shape: (87585, 20)


## 7. Resumen de valoraciones

In [31]:
# Calculamos la media y el numero de valoraciones por pelicula.
rating_summary = calculate_rating_summary(ratings_df)
display(rating_summary.head())
print('rating_summary shape:', rating_summary.shape)

,movieId,rating_mean,rating_count
0,1,3.897438,68997
1,2,3.275758,28904
2,3,3.139447,13134
3,4,2.845331,2806
4,5,3.059602,13154


rating_summary shape: (84432, 3)


## 8. Construccion de la tabla final

In [32]:
# Unimos peliculas, estadisticas de valoracion y generos codificados en una sola tabla.
movies_clean = build_movies_clean(movies_df, ratings_df)
display(movies_clean.head())
print('movies_clean shape:', movies_clean.shape)
print('movies_clean columns:', movies_clean.columns.tolist())

,movieId,title,genres,year,title_clean,rating_mean,rating_count,Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,3.897438,68997,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,3.275758,28904,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,3.139447,13134,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,2.845331,2806,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,3.059602,13154,0,0,0,...,0,0,0,0,0,0,0,0,0,0


movies_clean shape: (87585, 26)
movies_clean columns: ['movieId', 'title', 'genres', 'year', 'title_clean', 'rating_mean', 'rating_count', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


## 9. Guardado de resultados

In [33]:
processed_dir.mkdir(parents=True, exist_ok=True)

movies_clean.to_csv(processed_dir / 'movies_clean.csv', index=False)
movie_genres_encoded.to_csv(processed_dir / 'movie_genres_encoded.csv', index=False)

print('Archivos guardados en:', processed_dir)

Archivos guardados en: C:\Users\alexo\Desktop\TFG\data\processed


## 10. Cierre

Con este notebook ya tenemos una tabla limpia de peliculas con el ano extraido, el titulo normalizado, los generos codificados y un resumen de valoraciones. Esto deja los datos preparados para analisis exploratorio y para el siguiente paso del proyecto.